# Mini-Project 4 — A LangGraph Alert-Triage Agent
### IT7075: Applied AI for Cybersecurity · LangGraph
**Shamak Patel** · University of Cincinnati · Fall 2026

A chain runs straight through. An **agent decides** — and the decision is the interesting part,
because a triage agent can be wrong in two very different ways:

- it **misses** a real incident (the dangerous error), or
- it **escalates noise** and buries your analysts (the expensive error).

You will build the Lecture 3 triage graph, run it against **12 labelled alerts**, and measure both
kinds of error. Then you tune the rule and watch one error trade against the other.

**This project is mostly analysis, not coding.** There are five TODOs and they total **ten lines** —
the graph is small on purpose. Most of the credit is for what you write about the results.

| | LangGraph concept | You write |
|---|---|---|
| **TODO 1** | the **state** (a `TypedDict`) | 2 lines |
| **TODO 2** | a **node** (plain function → dict) | 2 lines |
| **TODO 3** | the **router** for a conditional edge | 1 line |
| **TODO 4** | a node that calls a **tool** | 2 lines |
| **TODO 5** | **build and compile** the graph | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs fully offline.** Every node is ordinary Python — no API key, no model download, no GPU.
> That is deliberate: the point is the *routing decision*, and a deterministic agent lets you
> measure it exactly.

## Part 1 — The state  ·  **TODO 1**

The state is the graph's memory: one dictionary that every node reads from and writes to. Declaring
it as a `TypedDict` is how you say *these* are the fields an alert carries as it moves through the
graph.

This is the first real difference from the LangChain module. There, data flowed along a pipe from
one step to the next. Here every node sees the **whole** state, and a node only returns the keys it
changed.

> **Expected output:** `state fields: ['alert', 'risk', 'cve_info', 'report']`.

In [1]:
import json, os
from typing import TypedDict

ALERTS_FILE = "alerts.json" if os.path.exists("alerts.json") else "../alerts.json"


###################  TODO 1 — declare the state  ###################
# WHAT: add the two missing fields to TriageState.
#
# The agent needs four things in its state, and two are already there:
#     alert     : str   the incoming alert text            (given)
#     risk      : str   "high" or "low", set by TODO 2     <- you add this
#     cve_info  : str   what the tool found, set by TODO 4 <- you add this
#     report    : str   the final summary                  (given)
#
# HOW (the Lecture 2 "Define the State" cell): inside the class, one line per
# field, in the form   name: type
#
# WHY a TypedDict and not a plain dict: it documents the agent's memory in one
# place, so when a node returns {"risk": "high"} you can see where that lands.
################

class TriageState(TypedDict):
    alert: str
    risk: str          # TODO 1: "high" or "low", set by classify
    cve_info: str      # TODO 1: what lookup_cve found, set by investigate
    report: str


print("state fields:", list(TriageState.__annotations__))

state fields: ['alert', 'risk', 'cve_info', 'report']


## Part 2 — The classify node  ·  **TODO 2**

A node is just a Python function: it takes the state and returns a dict of the keys it changed.

This particular node is the one your whole experiment turns on. It decides `risk` by looking for
words in the alert text — the same rule as the Lecture 3 notebook. Keep `KEYWORDS` as a variable:
Part 7 changes it and re-measures.

> **Expected output:** `high` for the RCE alert, `low` for the failed-login alert.

In [2]:
# The Lecture 3 keyword list. Part 7 will change this and re-measure.
KEYWORDS = ["rce", "ransomware", "exfil", "critical", "admin"]


###################  TODO 2 — write the classify node  ###################
# WHAT: finish classify() so it sets `risk` from the alert text.
#
# HOW (the Lecture 3 "Conditional Edges" cell):
#   * is any keyword present?  Search the LOWERCASED alert text:
#         any(word in state["alert"].lower() for word in KEYWORDS)
#   * a node returns a DICT of just the fields it changed -- here:
#         {"risk": "high"}  or  {"risk": "low"}
#     Do NOT return the whole state, and do not mutate it in place.
#
# NOTE: matching is substring-based, so "admin" also fires on "administrator".
#       Keep that in mind when you read your results -- it matters later.
################

def classify(state: TriageState) -> dict:
    hit = any(word in state["alert"].lower() for word in KEYWORDS)   # TODO 2
    return {"risk": "high" if hit else "low"}                         # TODO 2


# ---- given: try it on two alerts ----------------------------------------
for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    print("%-58s -> %s" % (text[:56], classify({"alert": text})["risk"]))

Critical RCE exploit attempt against the Apache server     -> high
Single failed login from a known corporate device          -> low


## Part 3 — The router  ·  **TODO 3**

This is what makes it a graph rather than a chain. A **router** is a function that looks at the
state and returns a *label* — the name of the branch to take next. LangGraph calls it after the
`classify` node and jumps to whichever node that label points at.

The router decides nothing else: no work, no side effects, just "which way".

> **Expected output:** `high -> investigate`, `low -> quick_close`.

In [3]:
###################  TODO 3 — write the router  ###################
# WHAT: return the LABEL of the next branch, based on state["risk"].
#
#     risk == "high"  ->  "investigate"
#     anything else   ->  "quick_close"
#
# HOW (the Lecture 3 route_by_risk function): one line, e.g.
#     return "investigate" if <condition> else "quick_close"
#
# These two strings are labels, not node names -- TODO 5 maps them to real
# nodes. Keeping them separate is what lets you rewire a branch without
# touching this function.
################

def route_by_risk(state: TriageState) -> str:
    return "investigate" if state["risk"] == "high" else "quick_close"       # TODO 3


# ---- given ---------------------------------------------------------------
for r in ["high", "low"]:
    print("%-5s -> %s" % (r, route_by_risk({"risk": r})))

high  -> investigate
low   -> quick_close


## Part 4 — A node that calls a tool  ·  **TODO 4**

A **tool** is just a function the agent can call for information it does not have. `lookup_cve` is
given — a tiny stand-in for a threat-intel API. Your job is the node that uses it and writes the
result into the state.

> **Expected output:** the RCE alert returns the `CVE-2026-0001` line; an unmatched alert returns
> the "No specific CVE found" line.

In [4]:
def lookup_cve(keyword: str) -> str:
    """GIVEN: the tool. A stand-in for a threat-intel lookup."""
    db = {"rce": "CVE-2026-0001: critical RCE; patch immediately.",
          "ransomware": "Multiple CVEs; isolate the host and restore from backup.",
          "credential": "Credential theft; force password resets and review logins.",
          "outbound": "Possible data theft; check egress logs and block the destination."}
    for key, value in db.items():
        if key in keyword.lower():
            return value
    return "No specific CVE found; follow standard triage."


###################  TODO 4 — write the investigate node  ###################
# WHAT: call the tool, then return BOTH fields it fills in.
#
# HOW:
#   * info = lookup_cve(state["alert"])
#   * return a dict with two keys:
#         "cve_info" -> info
#         "report"   -> "ESCALATE. " + info
#
# This is the whole idea of a tool: the node does not guess what the CVE is,
# it asks something that knows. Note the node returns two keys at once --
# a node may update as many state fields as it likes.
################

def investigate(state: TriageState) -> dict:
    info = lookup_cve(state["alert"])                                  # TODO 4
    return {"cve_info": info, "report": "ESCALATE. " + info}          # TODO 4


def quick_close(state: TriageState) -> dict:
    """GIVEN: the other branch. No tool call, no escalation."""
    return {"cve_info": "", "report": "CLOSE. Logged for monitoring; no action needed."}


# ---- given ---------------------------------------------------------------
for text in ["Critical RCE exploit attempt", "Backup job completed with a warning"]:
    print("%-32s -> %s" % (text, investigate({"alert": text})["cve_info"]))

Critical RCE exploit attempt     -> CVE-2026-0001: critical RCE; patch immediately.
Backup job completed with a warning -> No specific CVE found; follow standard triage.


## Part 5 — Build and compile the graph  ·  **TODO 5**

Now wire the pieces together. The nodes are added for you; you supply the three lines that make it
a runnable graph — where it starts, how the branch works, and the compile step.

```
        classify
           |  route_by_risk
     +-----+-----+
     |           |
investigate   quick_close
     |           |
     +-----+-----+
           END
```

> **Expected output:** the RCE alert reports `ESCALATE. CVE-2026-0001 ...`; the failed-login alert
> reports `CLOSE. Logged for monitoring ...`.

In [5]:
from langgraph.graph import StateGraph, END


def build_triage_graph():
    """Assemble the graph. Called again in Part 7 after KEYWORDS changes."""
    builder = StateGraph(TriageState)
    builder.add_node("classify", classify)
    builder.add_node("investigate", investigate)
    builder.add_node("quick_close", quick_close)

    ###################  TODO 5 — wire it up  ###################
    # WHAT: three lines, all from the Lecture 3 "Wire the Branching Graph" cell.
    #
    #   1. entry point -- the graph starts at the "classify" node:
    #          builder.set_entry_point("classify")
    #
    #   2. the conditional edge -- after "classify", call your router and map each
    #      LABEL it can return to the node that should run next:
    #          builder.add_conditional_edges("classify", route_by_risk,
    #                                        {"investigate": "investigate",
    #                                         "quick_close": "quick_close"})
    #      (left side = the label from TODO 3, right side = the node name)
    #
    #   3. compile it into a runnable graph and return it:
    #          return builder.compile()
    #
    # The two edges to END are already there: both branches finish the run.
    ################

    builder.set_entry_point("classify")                                # TODO 5
    builder.add_conditional_edges("classify", route_by_risk,           # TODO 5
                              {"investigate": "investigate",
                               "quick_close": "quick_close"})

    builder.add_edge("investigate", END)
    builder.add_edge("quick_close", END)
    return builder.compile()                                           # TODO 5


graph = build_triage_graph()


# ---- given: a blank state, then one run per alert ------------------------
def run(graph, alert_text):
    """GIVEN: invoke the graph on one alert and hand back the finished state."""
    return graph.invoke({"alert": alert_text, "risk": "", "cve_info": "", "report": ""})


for text in ["Critical RCE exploit attempt against the Apache server",
             "Single failed login from a known corporate device"]:
    out = run(graph, text)
    print("ALERT :", text)
    print("  risk:", out["risk"], "| report:", out["report"], "\n")

ALERT : Critical RCE exploit attempt against the Apache server
  risk: high | report: ESCALATE. CVE-2026-0001: critical RCE; patch immediately. 

ALERT : Single failed login from a known corporate device
  risk: low | report: CLOSE. Logged for monitoring; no action needed. 



## Part 6 — Score the agent  *(no TODO — run it and read it)*

The coding is done. Everything from here is measurement and interpretation.

`alerts.json` holds **12 real-looking alerts**, each labelled with what a SOC analyst would
actually do: **6 `escalate`** and **6 `close`**. Comparing your agent's route to that label gives
three numbers — and the two errors are not equally bad:

| | Meaning | Cost |
|---|---|---|
| **caught** | escalated something that deserved it | what you want |
| **missed** | closed a real incident | an attacker keeps going, unnoticed |
| **over-escalated** | escalated routine noise | analyst time, and eventually alert fatigue |

> **Expected output with the Lecture 3 keyword list: caught 3/6, missed 3/6, over-escalated 2/6.**
> The agent you just built misses **half** of the real incidents. Part 7 asks what to do about it.

In [6]:
ALERTS = json.load(open(ALERTS_FILE))["alerts"]
print("loaded", len(ALERTS), "labelled alerts\n")


def score(graph, alerts):
    """GIVEN: run every alert and compare the route with the analyst's label."""
    caught = missed = over = 0
    rows = []
    for a in alerts:
        out = run(graph, a["alert"])
        route = "escalate" if out["risk"] == "high" else "close"
        if a["expected"] == "escalate":
            verdict = "caught" if route == "escalate" else "MISSED"
        else:
            verdict = "over-escalated" if route == "escalate" else "closed ok"
        caught += verdict == "caught"
        missed += verdict == "MISSED"
        over += verdict == "over-escalated"
        rows.append({"id": a["id"], "expected": a["expected"], "agent": route,
                     "verdict": verdict, "alert": a["alert"][:52]})
    return caught, missed, over, rows


caught, missed, over, rows = score(graph, ALERTS)
n_esc = sum(a["expected"] == "escalate" for a in ALERTS)
n_close = len(ALERTS) - n_esc

print("caught          %d/%d real incidents" % (caught, n_esc))
print("MISSED          %d/%d real incidents      <-- the dangerous error" % (missed, n_esc))
print("over-escalated  %d/%d routine alerts      <-- the expensive error\n" % (over, n_close))

print("%-5s %-9s %-9s %-15s %s" % ("id", "expected", "agent", "verdict", "alert"))
for r in rows:
    print("%-5s %-9s %-9s %-15s %s" % (r["id"], r["expected"], r["agent"], r["verdict"], r["alert"]))

loaded 12 labelled alerts

caught          3/6 real incidents
MISSED          3/6 real incidents      <-- the dangerous error
over-escalated  2/6 routine alerts      <-- the expensive error

id    expected  agent     verdict         alert
A01   escalate  escalate  caught          Critical RCE exploit attempt against the internet-fa
A02   escalate  escalate  caught          Ransomware canary file modified on the finance file 
A03   escalate  close     MISSED          Large outbound data transfer to an unknown host at 0
A04   escalate  escalate  caught          New local administrator account created on the domai
A05   escalate  close     MISSED          Credential dumping tool signature detected in memory
A06   escalate  close     MISSED          Repeated MFA prompts followed by a successful login 
A07   close     escalate  over-escalated  Scheduled vulnerability scan reported critical findi
A08   close     close     closed ok       User reset their own password via the self-service p
A

## Part 7 — Tune the rule and watch the two errors trade  *(no TODO)*

The obvious fix for a missed incident is to add words. Do that four times over and watch what it
costs.

> **Expected output:**
>
> | keyword list | caught | missed | over-escalated |
> |---|---:|---:|---:|
> | narrow (2 words) | 2/6 | **4/6** | 0/6 |
> | **lecture (5 words)** | 3/6 | **3/6** | 2/6 |
> | wider (8 words) | **6/6** | 0/6 | 2/6 |
> | widest (11 words) | 6/6 | 0/6 | **4/6** |
>
> Read the two error columns against each other. Going from narrow to widest drives missed
> detections to zero and doubles the false escalations. There is no row with zeroes in both.
>
> **And look at *which* alerts stay wrong.** A07 and A12 are over-escalated by every list that
> catches all six real incidents. A07 says *critical* — about findings from our own scanner on a
> sandbox VM. A12 says *Administrator* — about an approved patch in a maintenance window. The
> trigger word is right there in both; what makes them harmless is **context the words do not
> carry**. No keyword list can fix that, which is the most important thing this table has to say.

In [7]:
KEYWORD_LISTS = {
    "narrow (2 words)":   ["rce", "ransomware"],
    "lecture (5 words)":  ["rce", "ransomware", "exfil", "critical", "admin"],
    "wider (8 words)":    ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa"],
    "widest (11 words)":  ["rce", "ransomware", "exfil", "critical", "admin",
                           "outbound", "credential", "mfa", "failed", "scan", "password"],
}

print("%-20s %-8s %-8s %-15s %s" % ("keyword list", "caught", "missed", "over-escalated", "which ones"))
sweep = []
for name, words in KEYWORD_LISTS.items():
    KEYWORDS[:] = words                      # the classify node reads this list
    c, m, o, rr = score(build_triage_graph(), ALERTS)
    bad = [r["id"] for r in rr if r["verdict"] == "MISSED"]
    fat = [r["id"] for r in rr if r["verdict"] == "over-escalated"]
    sweep.append({"list": name, "caught": c, "missed": m, "over": o})
    print("%-20s %-8s %-8s %-15s missed=%-14s over=%s"
          % (name, "%d/%d" % (c, n_esc), "%d/%d" % (m, n_esc), "%d/%d" % (o, n_close),
             ",".join(bad) or "none", ",".join(fat) or "none"))

KEYWORDS[:] = KEYWORD_LISTS["lecture (5 words)"]     # put it back
print("\nSaved to results.csv")
with open("results.csv", "w", encoding="utf-8") as f:
    f.write("keyword_list,caught,missed,over_escalated\n")
    for s in sweep:
        f.write("%s,%d,%d,%d\n" % (s["list"], s["caught"], s["missed"], s["over"]))

keyword list         caught   missed   over-escalated  which ones
narrow (2 words)     2/6      4/6      0/6             missed=A03,A04,A05,A06 over=none
lecture (5 words)    3/6      3/6      2/6             missed=A03,A05,A06    over=A07,A12
wider (8 words)      6/6      0/6      2/6             missed=none           over=A07,A12
widest (11 words)    6/6      0/6      4/6             missed=none           over=A07,A08,A10,A12

Saved to results.csv


## Part 8 — Memory: why the graph beats a chain  *(no TODO)*

One more LangGraph feature from Lecture 3. Compile the **same** graph with a `MemorySaver` and every
run tagged with a `thread_id` is remembered, so an incident can be picked up later — something a
one-shot chain cannot do.

> **Expected output:** the state saved under `incident-42` is still readable after the run, showing
> the alert, the risk, and the report the agent produced.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(TriageState)
builder.add_node("classify", classify)
builder.add_node("investigate", investigate)
builder.add_node("quick_close", quick_close)
builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route_by_risk,
                              {"investigate": "investigate", "quick_close": "quick_close"})
builder.add_edge("investigate", END)
builder.add_edge("quick_close", END)
graph_with_memory = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "incident-42"}}
out = graph_with_memory.invoke(
    {"alert": "Ransomware canary file modified on the finance file share",
     "risk": "", "cve_info": "", "report": ""}, config)
print("run     :", out["report"])

# ...later, or in another cell, the incident is still there:
saved = graph_with_memory.get_state(config).values
print("recalled:", saved["alert"])
print("          risk =", saved["risk"], "| cve_info =", saved["cve_info"][:48])

print("\nA different thread_id starts a fresh incident:")
other = {"configurable": {"thread_id": "incident-99"}}
print("  incident-99 has state yet?", bool(graph_with_memory.get_state(other).values))

run     : ESCALATE. Multiple CVEs; isolate the host and restore from backup.
recalled: Ransomware canary file modified on the finance file share
          risk = high | cve_info = Multiple CVEs; isolate the host and restore from

A different thread_id starts a fresh incident:
  incident-99 has state yet? False


## Part 9: your own alerts, and your own change to the graph

Everything above ran on the provided alerts, and it was practice. A rule tuned on twelve examples
someone else wrote proves very little, so the rest of this notebook is your own.

First write the data. Copy `alerts_template.json` to `my_alerts.json` and write 12 labelled alerts,
6 you would escalate and 6 you would close. Base them on real alert types such as Windows event
logs, EDR detections, cloud audit trails, or IDS signatures, describe them in your own words, and
put any sources in `SOURCES.md`. Include at least two hard cases: one real incident whose text
contains none of your keywords, and one routine alert that does contain one. For every alert, be
able to say why an analyst would escalate or close it.

No code is provided from here on. Add cells below that:

1. Load your alerts and report how many of each label you have.
2. Run the agent on them and score the three outcomes: caught, missed, and over escalated.
3. Change the rule a few times, one change at a time, and record what each change costs on the two
   error columns. Save the table as `results_mine.csv`.
4. Make at least one change to the graph itself, not just the word list. Add a node, add a tool,
   change the router, or add a second decision, then measure the same alerts again and say what it
   bought you.

> When you are finished, run the whole notebook top to bottom and save it with the outputs showing,
> then commit it to your private GitHub repository along with `my_alerts.json` and your results.
> See `MiniProject_Agent.md` section 6.1 for the folder structure.

In [9]:
###################  YOUR ALERTS  ###################
# Load my_alerts.json, report the label counts, run the agent, and score
# caught, missed, and over escalated.
################
from collections import Counter

MY = json.load(open("my_alerts.json"))
MY_ALERTS = MY["alerts"]
print("loaded %d of my alerts: %s" % (len(MY_ALERTS), dict(Counter(a["expected"] for a in MY_ALERTS))))
n_esc_mine = sum(a["expected"] == "escalate" for a in MY_ALERTS)
n_close_mine = len(MY_ALERTS) - n_esc_mine

# check the hard cases really are hard for the lecture rule
LECTURE = ["rce", "ransomware", "exfil", "critical", "admin"]
by_id = {a["id"]: a for a in MY_ALERTS}
for aid in MY["hard_cases"]["escalate_without_keywords"]:
    hits = [w for w in LECTURE if w in by_id[aid]["alert"].lower()]
    print("hard escalate %s  lecture keywords in text: %s" % (aid, hits or "none"))
for aid in MY["hard_cases"]["close_with_a_keyword"]:
    hits = [w for w in LECTURE if w in by_id[aid]["alert"].lower()]
    print("hard close    %s  lecture keywords in text: %s" % (aid, hits))

KEYWORDS[:] = LECTURE
caught, missed, over, rows = score(build_triage_graph(), MY_ALERTS)
print("\nThe Lecture 3 agent, unchanged, on my alerts:")
print("caught          %d/%d real incidents" % (caught, n_esc_mine))
print("MISSED          %d/%d real incidents      <-- the dangerous error" % (missed, n_esc_mine))
print("over-escalated  %d/%d routine alerts      <-- the expensive error\n" % (over, n_close_mine))
print("%-5s %-9s %-9s %-15s %s" % ("id", "expected", "agent", "verdict", "alert"))
for r in rows:
    print("%-5s %-9s %-9s %-15s %s" % (r["id"], r["expected"], r["agent"], r["verdict"], r["alert"]))

loaded 12 of my alerts: {'escalate': 6, 'close': 6}
hard escalate M02  lecture keywords in text: none
hard escalate M04  lecture keywords in text: none
hard escalate M05  lecture keywords in text: none
hard escalate M06  lecture keywords in text: none
hard close    M07  lecture keywords in text: ['critical']
hard close    M08  lecture keywords in text: ['admin']
hard close    M12  lecture keywords in text: ['exfil']

The Lecture 3 agent, unchanged, on my alerts:
caught          2/6 real incidents
MISSED          4/6 real incidents      <-- the dangerous error
over-escalated  3/6 routine alerts      <-- the expensive error

id    expected  agent     verdict         alert
M01   escalate  escalate  caught          EDR on FIN-FS01: ransomware behaviour blocked after 
M02   escalate  close     MISSED          Windows event 1102 on DC01: the Security audit log w
M03   escalate  escalate  caught          CloudTrail: IAM user ci-deploy attached Administrato
M04   escalate  close     MISSED    

### Tracing one alert through the graph

`stream(..., stream_mode="updates")` prints each node as it runs and the keys it returned, so you can
watch the state move. M01 is **caught**: `classify` finds *ransomware*, the router sends it to
`investigate`, which calls the tool. M04 is **missed**: none of the five words is in the text, so
the router sends a malicious-macro chain straight to `quick_close`.

In [10]:
def trace(graph, state):
    """Run one alert and print every node that ran and the fields it returned."""
    print("ALERT:", state["alert"])
    for step in graph.stream(state, stream_mode="updates"):
        for node, update in step.items():
            print("  -> %-18s returned %s" % (node, update))
    print()


KEYWORDS[:] = LECTURE
lecture_graph = build_triage_graph()
for aid in ["M01", "M04"]:
    trace(lecture_graph, {"alert": by_id[aid]["alert"], "risk": "", "cve_info": "", "report": ""})

ALERT: EDR on FIN-FS01: ransomware behaviour blocked after 3,400 files on the finance share were renamed to .lockd in two minutes
  -> classify           returned {'risk': 'high'}
  -> investigate        returned {'cve_info': 'Multiple CVEs; isolate the host and restore from backup.', 'report': 'ESCALATE. Multiple CVEs; isolate the host and restore from backup.'}

ALERT: EDR process tree on HR-LT-14: WINWORD.EXE started powershell.exe with an encoded command that then connected to a domain registered two days ago
  -> classify           returned {'risk': 'low'}
  -> quick_close        returned {'cve_info': '', 'report': 'CLOSE. Logged for monitoring; no action needed.'}



### Your tuning and your change to the graph

**Step 1: the word list, one change at a time.** Each row changes only the keyword list; the graph
is the same Lecture 3 graph. I add words for exactly the incidents the previous row missed, which is
what an analyst would do after a miss, and then count what the new words cost.

In [11]:
###################  YOUR TUNING AND YOUR GRAPH CHANGE  ###################
# Change the rule one step at a time and record what each change costs.
# Then change the graph itself and measure the same alerts again.
# Save the table as results_mine.csv.
################
MY_KEYWORD_LISTS = {
    "narrow (2 words)":  ["rce", "ransomware"],
    "lecture (5 words)": LECTURE,
    "wider (8 words)":   LECTURE + ["cleared", "powershell", "encoded"],           # for M02, M04
    "widest (13 words)": LECTURE + ["cleared", "powershell", "encoded",
                                    "travel", "dns", "password", "scan", "test"],  # for M05, M06
}
WHY = {
    "narrow (2 words)":  "remove exfil, critical, admin from the lecture list",
    "lecture (5 words)": "baseline: the Lecture 3 rule",
    "wider (8 words)":   "add cleared, powershell, encoded (for missed M02, M04)",
    "widest (13 words)": "add travel, dns, password, scan, test (for missed M05, M06)",
}

results = []      # every measured setting goes here, then into results_mine.csv


def record(step, words_name, graph_name, change, c, m, o, rr):
    results.append({
        "step": step, "keyword_list": words_name, "graph": graph_name, "one_change": change,
        "caught": c, "missed": m, "over_escalated": o, "closed_ok": n_close_mine - o,
        "missed_ids": " ".join(r["id"] for r in rr if r["verdict"] == "MISSED") or "none",
        "over_ids": " ".join(r["id"] for r in rr if r["verdict"] == "over-escalated") or "none"})
    print("%-2d %-19s %-34s caught %d/%d  missed %d/%d  over %d/%d   missed=%-15s over=%s"
          % (step, words_name, graph_name, c, n_esc_mine, m, n_esc_mine, o, n_close_mine,
             results[-1]["missed_ids"], results[-1]["over_ids"]))


for step, (name, words) in enumerate(MY_KEYWORD_LISTS.items(), start=1):
    KEYWORDS[:] = words
    c, m, o, rr = score(build_triage_graph(), MY_ALERTS)
    record(step, name, "lecture graph", WHY[name], c, m, o, rr)
KEYWORDS[:] = LECTURE

1  narrow (2 words)    lecture graph                      caught 1/6  missed 5/6  over 0/6   missed=M02 M03 M04 M05 M06 over=none
2  lecture (5 words)   lecture graph                      caught 2/6  missed 4/6  over 3/6   missed=M02 M04 M05 M06 over=M07 M08 M12
3  wider (8 words)     lecture graph                      caught 4/6  missed 2/6  over 3/6   missed=M05 M06         over=M07 M08 M12
4  widest (13 words)   lecture graph                      caught 6/6  missed 0/6  over 5/6   missed=none            over=M07 M08 M09 M10 M12


Read the two error columns against each other. Every word I add to stop a miss also fires on
routine alerts: *scan* and *test* hit my own scanner and the EICAR check, *password* hits an
ordinary lockout. Going from the lecture list to the widest list drives missed incidents from 4 to 0
and raises over-escalations from 3 to 5. No list gets both columns to zero.

Three alerts stay wrong at **every** list that catches more than one incident: **M07** (*critical*,
on a lab host), **M08** (*Administrators*, under an approved change) and **M12** (*exfiltration*,
the nightly backup). The trigger word really is in each one. What makes them harmless is **which
host** it happened on and **whether a change was approved**, and no word list can know either. The
fix has to come from outside the alert text, which means changing the graph.

**Step 2: change the graph itself.** I add a `context` node that runs after `classify`, two tools
it calls, and a new router with new branches:

| Tool (plain Python, like `lookup_cve`) | What it answers |
|---|---|
| `lookup_asset(host)` | the asset inventory: is this host production, lab, or honeypot, and how critical? |
| `check_change(ticket, host)` | the change calendar: does the cited ticket exist, is it approved, and is it for **this** host? |

The new router replaces `route_by_risk`. A keyword hit still means "high risk", but the router now
asks two more questions before it escalates: a hit on a **non-production** asset, or a hit that
matches an **approved change on that same host**, goes to a new `close_with_context` node, whose
report says why. Every other keyword hit goes to `investigate` exactly as before.

Two rules keep this fail-safe: a host the inventory does not know counts as production, and a
ticket that does not exist, is not approved, or names a different host does **not** close anything.

A third, optional change makes the router read **memory**: with `use_memory`, an alert with no
keyword that arrives on a thread which already holds an escalation goes to an `add_to_case` branch
and is escalated as part of that case. It only matters when alerts share a `thread_id`; it is
demonstrated at the end.

Each change is switched on as its own measured step.

In [12]:
import operator
import re
from typing import Annotated


class MyTriageState(TriageState):
    """The Lecture 3 state plus what the context node learns."""
    host: str                                # the host named in the alert
    asset: dict                              # lookup_asset's answer
    change: dict                             # check_change's answer
    history: Annotated[list, operator.add]   # every alert seen on this thread (appended, not replaced)
    escalations: Annotated[list, operator.add]   # every alert escalated on this thread


# ---- tool 1: the asset inventory (a stand-in for a CMDB) -----------------------------------
ASSETS = {
    "FIN-FS01":  {"env": "production", "role": "finance file server",           "criticality": "high"},
    "DC01":      {"env": "production", "role": "domain controller",             "criticality": "crown jewel"},
    "HR-LT-14":  {"env": "production", "role": "HR laptop",                     "criticality": "medium"},
    "PAY-DB02":  {"env": "production", "role": "payments database",             "criticality": "crown jewel"},
    "SCAN01":    {"env": "production", "role": "our vulnerability scanner",     "criticality": "medium"},
    "LAB-WEB03": {"env": "lab",        "role": "lab web server, rebuilt nightly", "criticality": "low"},
    "APP-SRV07": {"env": "production", "role": "application server",            "criticality": "medium"},
    "IT-LT-02":  {"env": "production", "role": "IT staff laptop",               "criticality": "medium"},
    "HP-01":     {"env": "honeypot",   "role": "internet-facing honeypot",      "criticality": "low"},
    "BKP-01":    {"env": "production", "role": "backup server",                 "criticality": "high"},
}
NON_PRODUCTION = {"lab", "honeypot"}


def lookup_asset(host: str) -> dict:
    """TOOL. Unknown hosts are treated as production: never assume something is safe."""
    return ASSETS.get(host, {"env": "production", "role": "not in inventory", "criticality": "unknown"})


# ---- tool 2: the change calendar ------------------------------------------------------------
CHANGES = {
    "CHG-20417": {"status": "approved", "hosts": ["APP-SRV07"],
                  "what": "SQL Server install; add svc_sql2 to local Administrators"},
    "CHG-20450": {"status": "approved", "hosts": ["IT-LT-02"],
                  "what": "monthly antivirus check with the EICAR test file"},
    "CHG-19001": {"status": "approved", "hosts": ["BKP-01"],
                  "what": "standing change: nightly offsite backup to the contracted provider"},
    "CHG-20502": {"status": "rejected", "hosts": ["DC01"],
                  "what": "add helpdesk group to Domain Admins (rejected by CAB)"},
}


def check_change(ticket: str, host: str) -> dict:
    """TOOL. A ticket only counts if it exists, is approved, and covers this host."""
    if not ticket:
        return {"ticket": "", "approved": False, "why": "no change ticket cited"}
    rec = CHANGES.get(ticket)
    if rec is None:
        return {"ticket": ticket, "approved": False, "why": "ticket not found in the change calendar"}
    if rec["status"] != "approved":
        return {"ticket": ticket, "approved": False, "why": "ticket is " + rec["status"]}
    if host not in rec["hosts"]:
        return {"ticket": ticket, "approved": False,
                "why": "ticket covers %s, not %s" % (", ".join(rec["hosts"]), host or "an unnamed host")}
    return {"ticket": ticket, "approved": True, "why": "approved change on this host: " + rec["what"]}


# ---- pulling the host and ticket out of the alert text ---------------------------------------
HOST = r"\b(?!CHG-)[A-Z]{2,}(?:-[A-Z]{2,})?-?\d{2}\b"
HOST_RE = re.compile(HOST)
TARGET_RE = re.compile(r"\b(?:on|against)\s+(" + HOST + ")")   # "... on DC01", "... against LAB-WEB03"
TICKET_RE = re.compile(r"\bCHG-\d{5}\b")


def extract_host(text: str, rule: str = "target") -> str:
    """rule='first': the first host named anywhere (my first version).
    rule='target': the host the alert happened ON, falling back to the first host named."""
    if rule == "target":
        m = TARGET_RE.search(text)
        if m:
            return m.group(1)
    m = HOST_RE.search(text)
    return m.group(0) if m else ""


def extract_ticket(text: str) -> str:
    m = TICKET_RE.search(text)
    return m.group(0) if m else ""


# ---- the new node, router, and branch --------------------------------------------------------
def make_context_node(use_asset: bool, use_change: bool, host_rule: str):
    """The context node calls whichever tools are switched on and writes the answers to state."""
    def context(state: MyTriageState) -> dict:
        host = extract_host(state["alert"], host_rule)
        update = {"host": host, "history": [state["alert"]]}
        if use_asset:
            update["asset"] = lookup_asset(host)
        if use_change:
            update["change"] = check_change(extract_ticket(state["alert"]), host)
        return update
    return context


def make_router(use_memory: bool):
    def route_with_context(state: MyTriageState) -> str:
        """The new router: a keyword hit is escalated unless context explains it.
        With use_memory, a quiet alert on a thread that already holds an escalation joins that case."""
        if state["risk"] != "high":
            if use_memory and state.get("escalations"):
                return "add_to_case"
            return "quick_close"
        if state.get("asset", {}).get("env") in NON_PRODUCTION:
            return "close_with_context"
        if state.get("change", {}).get("approved"):
            return "close_with_context"
        return "investigate"
    return route_with_context


def investigate_and_log(state: MyTriageState) -> dict:
    """The TODO 4 node, unchanged, plus a note in the thread's list of escalations."""
    return {**investigate(state), "escalations": [state["alert"]]}


def add_to_case(state: MyTriageState) -> dict:
    """Memory branch: no keyword, but this thread already has an escalated alert."""
    n = len(state["escalations"])
    return {"cve_info": "", "escalations": [state["alert"]],
            "report": "ESCALATE (open case). No keyword, but this thread already holds "
                      "%d escalated alert(s): %s" % (n, state["escalations"][0][:50])}


def close_with_context(state: MyTriageState) -> dict:
    """The new branch: closed, but the report says which piece of context closed it."""
    asset, change = state.get("asset", {}), state.get("change", {})
    if asset.get("env") in NON_PRODUCTION:
        why = "%s is a %s host (%s)" % (state["host"], asset["env"], asset["role"])
    else:
        why = "%s, %s" % (change["ticket"], change["why"])
    return {"cve_info": "", "report": "CLOSE (context). Keyword hit explained: " + why + "."}


def build_my_graph(use_asset=False, use_change=False, use_memory=False,
                   host_rule="target", checkpointer=None):
    """The Lecture 3 graph, plus my context node and new router when any of my changes is on."""
    b = StateGraph(MyTriageState)
    b.add_node("classify", classify)
    b.add_node("investigate", investigate_and_log)
    b.add_node("quick_close", quick_close)
    b.set_entry_point("classify")
    if use_asset or use_change or use_memory:
        b.add_node("context", make_context_node(use_asset, use_change, host_rule))
        b.add_node("close_with_context", close_with_context)
        b.add_edge("classify", "context")
        branches = {"investigate": "investigate", "quick_close": "quick_close",
                    "close_with_context": "close_with_context"}
        if use_memory:
            b.add_node("add_to_case", add_to_case)
            b.add_edge("add_to_case", END)
            branches["add_to_case"] = "add_to_case"
        b.add_conditional_edges("context", make_router(use_memory), branches)
        b.add_edge("close_with_context", END)
    else:
        b.add_conditional_edges("classify", route_by_risk,
                                {"investigate": "investigate", "quick_close": "quick_close"})
    b.add_edge("investigate", END)
    b.add_edge("quick_close", END)
    return b.compile(checkpointer=checkpointer)


def blank(text):
    return {"alert": text, "risk": "", "cve_info": "", "report": "",
            "host": "", "asset": {}, "change": {}, "history": [], "escalations": []}


def score_mine(graph, alerts):
    """Like score(), but the route is read from the branch that ran, not from `risk`:
    with the new router a high-risk alert can still be closed."""
    caught = missed = over = 0
    rows = []
    for a in alerts:
        out = graph.invoke(blank(a["alert"]))
        route = "escalate" if out["report"].startswith("ESCALATE") else "close"
        if a["expected"] == "escalate":
            verdict = "caught" if route == "escalate" else "MISSED"
        else:
            verdict = "over-escalated" if route == "escalate" else "closed ok"
        caught += verdict == "caught"
        missed += verdict == "MISSED"
        over += verdict == "over-escalated"
        rows.append({"id": a["id"], "expected": a["expected"], "agent": route, "verdict": verdict,
                     "host": out["host"], "report": out["report"][:70]})
    return caught, missed, over, rows


# sanity check: with both tools off, my graph must score exactly like the lecture graph
for name, words in MY_KEYWORD_LISTS.items():
    KEYWORDS[:] = words
    assert score_mine(build_my_graph(), MY_ALERTS)[:3] == score(build_triage_graph(), MY_ALERTS)[:3]
KEYWORDS[:] = LECTURE
print("check passed: with no tools switched on, build_my_graph() scores the same as the lecture graph\n")

print("hosts and tickets the context node finds, under each host rule:")
print("  id   %-24s %-24s ticket" % ("first host named (env)", "host it happened on (env)"))
for a in MY_ALERTS:
    h1, h2, t = extract_host(a["alert"], "first"), extract_host(a["alert"], "target"), extract_ticket(a["alert"])
    flag = "   <-- differ" if h1 != h2 else ""
    print("  %s  %-24s %-24s %s%s" % (a["id"], "%s (%s)" % (h1 or "-", lookup_asset(h1)["env"]),
          "%s (%s)" % (h2 or "-", lookup_asset(h2)["env"]), t or "-", flag))

print()
print(build_my_graph(use_asset=True, use_change=True, use_memory=True).get_graph().draw_ascii())

check passed: with no tools switched on, build_my_graph() scores the same as the lecture graph

hosts and tickets the context node finds, under each host rule:
  id   first host named (env)   host it happened on (env) ticket
  M01  FIN-FS01 (production)    FIN-FS01 (production)    -
  M02  DC01 (production)        DC01 (production)        -
  M03  - (production)           - (production)           -
  M04  HR-LT-14 (production)    HR-LT-14 (production)    -
  M05  - (production)           - (production)           -
  M06  PAY-DB02 (production)    PAY-DB02 (production)    -
  M07  SCAN01 (production)      LAB-WEB03 (lab)          -   <-- differ
  M08  APP-SRV07 (production)   APP-SRV07 (production)   CHG-20417
  M09  IT-LT-02 (production)    IT-LT-02 (production)    CHG-20450
  M10  - (production)           - (production)           -
  M11  HP-01 (honeypot)         HP-01 (honeypot)         -
  M12  BKP-01 (production)      BKP-01 (production)      CHG-19001



                                            +-----------+                                            
                                            | __start__ |                                            
                                            +-----------+                                            
                                                  *                                                  
                                                  *                                                  
                                                  *                                                  
                                            +----------+                                             
                                            | classify |                                             
                                            +----------+                                             
                                                  *                               

Now measure the same 12 alerts, one change per row, continuing the table from Step 1:

* **row 5** keeps the widest list (the only list that catches all six incidents) and adds the
  context node with `lookup_asset` and the new router. My first version took the **first** host
  named in the alert;
* **row 6** changes only that: the host is the one the alert happened **on**;
* **row 7** adds `check_change`, the second decision;
* **rows 8 and 9** keep that graph and narrow the word list back down, to see whether the graph lets
  a shorter list do the job;
* **row 10** turns on the memory branch. Every alert here runs on its own fresh thread, so this row
  checks that the branch costs nothing on independent alerts.

In [13]:
graph_steps = [
    (5,  "widest (13 words)", "+ lookup_asset (first host named)", dict(use_asset=True, host_rule="first"),
         "graph: add context node, lookup_asset tool, new router"),
    (6,  "widest (13 words)", "+ lookup_asset (host it is on)",    dict(use_asset=True),
         "host = the asset the alert happened on, not the first named"),
    (7,  "widest (13 words)", "+ check_change (both tools)",       dict(use_asset=True, use_change=True),
         "graph: add check_change tool, a second decision"),
    (8,  "wider (8 words)",   "+ check_change (both tools)",       dict(use_asset=True, use_change=True),
         "word list back to wider; graph as in row 7"),
    (9,  "lecture (5 words)", "+ check_change (both tools)",       dict(use_asset=True, use_change=True),
         "word list back to lecture; graph as in row 7"),
    (10, "widest (13 words)", "+ router reads thread memory",      dict(use_asset=True, use_change=True, use_memory=True),
         "graph: memory branch on (each alert on its own thread)"),
]
detail = {}
for step, words_name, graph_name, flags, change in graph_steps:
    KEYWORDS[:] = MY_KEYWORD_LISTS[words_name]
    c, m, o, rr = score_mine(build_my_graph(**flags), MY_ALERTS)
    record(step, words_name, graph_name, change, c, m, o, rr)
    detail[step] = rr
KEYWORDS[:] = LECTURE

import csv
with open("results_mine.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0]))
    w.writeheader()
    w.writerows(results)
print("\nSaved %d rows to results_mine.csv" % len(results))

print("\nRow 7, alert by alert:")
print("%-4s %-9s %-9s %-15s %-10s %s" % ("id", "expected", "agent", "verdict", "host", "report"))
for r in detail[7]:
    print("%-4s %-9s %-9s %-15s %-10s %s" % (r["id"], r["expected"], r["agent"], r["verdict"], r["host"], r["report"]))

5  widest (13 words)   + lookup_asset (first host named)  caught 6/6  missed 0/6  over 5/6   missed=none            over=M07 M08 M09 M10 M12
6  widest (13 words)   + lookup_asset (host it is on)     caught 6/6  missed 0/6  over 4/6   missed=none            over=M08 M09 M10 M12
7  widest (13 words)   + check_change (both tools)        caught 6/6  missed 0/6  over 1/6   missed=none            over=M10
8  wider (8 words)     + check_change (both tools)        caught 4/6  missed 2/6  over 0/6   missed=M05 M06         over=none
9  lecture (5 words)   + check_change (both tools)        caught 2/6  missed 4/6  over 0/6   missed=M02 M04 M05 M06 over=none
10 widest (13 words)   + router reads thread memory       caught 6/6  missed 0/6  over 1/6   missed=none            over=M10



Saved 10 rows to results_mine.csv

Row 7, alert by alert:
id   expected  agent     verdict         host       report
M01  escalate  escalate  caught          FIN-FS01   ESCALATE. Multiple CVEs; isolate the host and restore from backup.
M02  escalate  escalate  caught          DC01       ESCALATE. No specific CVE found; follow standard triage.
M03  escalate  escalate  caught                     ESCALATE. No specific CVE found; follow standard triage.
M04  escalate  escalate  caught          HR-LT-14   ESCALATE. No specific CVE found; follow standard triage.
M05  escalate  escalate  caught                     ESCALATE. No specific CVE found; follow standard triage.
M06  escalate  escalate  caught          PAY-DB02   ESCALATE. No specific CVE found; follow standard triage.
M07  close     close     closed ok       LAB-WEB03  CLOSE (context). Keyword hit explained: LAB-WEB03 is a lab host (lab w
M08  close     close     closed ok       APP-SRV07  CLOSE (context). Keyword hit explained: CHG

**What the graph change bought.** Between row 4 and row 7 the word list did not change, yet
over-escalations fell from 5 to 1 and no incident was lost.

* **Row 5 bought nothing**, and the reason is worth knowing. M07 names two hosts: `SCAN01`, our
  scanner, which *sends* the scan, and `LAB-WEB03`, the lab host it scanned. My first context node
  took the first host named, so it asked the inventory about the scanner, which is production. The
  tool worked; the field it was given was wrong.
* **Row 6** fixes only that: the host is the one the alert happened *on*. `lookup_asset` now sees
  the lab host and closes M07.
* **Row 7**: `check_change` closes M08, M09 and M12, each because a real, approved ticket names that
  exact host.
* **Rows 8 and 9**: with the graph in place the shorter lists over-escalate nothing, but they still
  miss what their words cannot see. Context only ever *closes* a keyword hit, so it fixes
  over-escalation and cannot fix a miss.
* **Row 10**: the memory branch changes nothing on twelve independent alerts, as it should.

What is left wrong at my chosen setting (row 7) is **M10**, a routine lockout that trips *password*.
It has no host and no ticket, so the context has nothing to say about it. That is the alert I am
choosing to get wrong (see the report).

The traces below show the new branch on three alerts: an approved change (M08), the lab host (M07),
and the one that still over-escalates (M10).

In [14]:
KEYWORDS[:] = MY_KEYWORD_LISTS["widest (13 words)"]
final_graph = build_my_graph(use_asset=True, use_change=True)
for aid in ["M08", "M07", "M10"]:
    trace(final_graph, blank(by_id[aid]["alert"]))
KEYWORDS[:] = LECTURE

ALERT: Windows event 4732: svc_sql2 was added to the local Administrators group on APP-SRV07 under change CHG-20417
  -> classify           returned {'risk': 'high'}
  -> context            returned {'host': 'APP-SRV07', 'history': ['Windows event 4732: svc_sql2 was added to the local Administrators group on APP-SRV07 under change CHG-20417'], 'asset': {'env': 'production', 'role': 'application server', 'criticality': 'medium'}, 'change': {'ticket': 'CHG-20417', 'approved': True, 'why': 'approved change on this host: SQL Server install; add svc_sql2 to local Administrators'}}
  -> close_with_context returned {'cve_info': '', 'report': 'CLOSE (context). Keyword hit explained: CHG-20417, approved change on this host: SQL Server install; add svc_sql2 to local Administrators.'}

ALERT: Nessus scheduled scan from SCAN01 reported 4 critical findings on LAB-WEB03, a lab web server rebuilt every night
  -> classify           returned {'risk': 'high'}
  -> context            returned {'host': '

### Can the change check be fooled?

A rule that closes alerts is something an attacker would love to trigger. The text *approved* or
*change* alone closes nothing; only a ticket the change calendar confirms for that host does. These
four extra alerts are not part of my 12 and are not scored. They test the new decision on purpose.
The last one shows the **cost** of the lab rule: an RCE on a lab host is closed, which is only safe
if the lab really is isolated from production.

In [15]:
KEYWORDS[:] = MY_KEYWORD_LISTS["widest (13 words)"]
probes = [
    "Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-20417",   # real ticket, wrong host
    "Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-20502",   # ticket was rejected
    "Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-77777",   # ticket does not exist
    "Critical RCE exploit attempt against LAB-WEB03, approved and expected",                      # lab rule closes it
]
for text in probes:
    out = final_graph.invoke(blank(text))
    print(text)
    print("   change check:", out["change"].get("why"), "| asset env:", out["asset"].get("env"))
    print("   ->", out["report"][:90], "\n")
KEYWORDS[:] = LECTURE

Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-20417
   change check: ticket covers APP-SRV07, not DC01 | asset env: production
   -> ESCALATE. No specific CVE found; follow standard triage. 

Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-20502
   change check: ticket is rejected | asset env: production
   -> ESCALATE. No specific CVE found; follow standard triage. 

Windows event 4732: helpdesk01 was added to Domain Admins on DC01 under change CHG-77777
   change check: ticket not found in the change calendar | asset env: production
   -> ESCALATE. No specific CVE found; follow standard triage. 

Critical RCE exploit attempt against LAB-WEB03, approved and expected
   change check: no change ticket cited | asset env: lab
   -> CLOSE (context). Keyword hit explained: LAB-WEB03 is a lab host (lab web server, rebuilt n 



### Does the change carry over to the provided alerts?

I ran the finished graph on the course's `alerts.json` with the course's widest list. A07 (*our own
scanner on the sandbox VM*) and A12 (*approved patch in the maintenance window*) are the same kind of
alert that context fixed in mine, but their text names no host and no ticket, so the tools have
nothing to look up and the score does not move. The improvement comes from the **context the alert
carries**, not from my rule fitting my own data. That is also a requirement for deploying it: the
alert pipeline must pass the host and ticket along.

In [16]:
KEYWORDS[:] = KEYWORD_LISTS["widest (11 words)"]
c0, m0, o0, _ = score(build_triage_graph(), ALERTS)
c1, m1, o1, rr1 = score_mine(build_my_graph(True, True), ALERTS)
print("course alerts, widest list, lecture graph : caught %d/6  missed %d/6  over %d/6" % (c0, m0, o0))
print("course alerts, widest list, my graph      : caught %d/6  missed %d/6  over %d/6" % (c1, m1, o1))
for r in rr1:
    if r["verdict"] == "over-escalated":
        print("   still over-escalated: %s  host found=%r" % (r["id"], r["host"]))
KEYWORDS[:] = LECTURE

course alerts, widest list, lecture graph : caught 6/6  missed 0/6  over 4/6
course alerts, widest list, my graph      : caught 6/6  missed 0/6  over 4/6
   still over-escalated: A07  host found=''
   still over-escalated: A08  host found=''
   still over-escalated: A10  host found=''
   still over-escalated: A12  host found=''


### Memory by thread id, on my own alert

Compile the finished graph with a `MemorySaver`, and every run tagged with a `thread_id` is saved.
I use one thread per **case**. M05 (impossible travel for a.khan) opens case `case-a.khan`. Later a
second alert about the same account arrives: a new inbox rule forwarding a.khan's mail outside the
company, which is what an attacker does next with a stolen mailbox. It contains **no keyword**.

I run it three ways:

1. on the same thread, with the memory branch **off**: the case remembers both alerts, but the
   router ignores that and closes the follow-up;
2. on the same thread, with the memory branch **on**: the router reads the thread's `escalations`
   and adds the follow-up to the open case;
3. the same follow-up on a **new** thread, memory branch on: nothing to remember, so it is closed.

`history` and `escalations` are declared with the `operator.add` reducer, so a second run on a thread
**appends** to them instead of replacing them. A chain has nowhere to keep this: each call starts
from nothing.

In [17]:
from langgraph.checkpoint.memory import MemorySaver

KEYWORDS[:] = MY_KEYWORD_LISTS["widest (13 words)"]
follow_up = ("Exchange audit: new inbox rule on a.khan's mailbox forwards all mail to an external "
             "address, created from the same Lagos session")


def open_case_then_follow_up(graph, thread_id):
    case = {"configurable": {"thread_id": thread_id}}
    first = graph.invoke(blank(by_id["M05"]["alert"]), case)
    # ...later. Only the new alert is passed in; the reducers append to what the thread saved.
    second = graph.invoke({"alert": follow_up, "risk": "", "cve_info": "", "report": ""}, case)
    return case, first, second


print("1) same thread, memory branch OFF")
mem_off = build_my_graph(use_asset=True, use_change=True, checkpointer=MemorySaver())
case, r1, r2 = open_case_then_follow_up(mem_off, "case-a.khan")
print("   M05       ->", r1["report"])
print("   follow-up ->", r2["report"])
saved = mem_off.get_state(case).values
print("   the thread remembers %d alerts:" % len(saved["history"]))
for i, a in enumerate(saved["history"], 1):
    print("     %d. %s" % (i, a[:95]))

print("\n2) same thread, memory branch ON")
mem_on = build_my_graph(use_asset=True, use_change=True, use_memory=True, checkpointer=MemorySaver())
case, r1, r2 = open_case_then_follow_up(mem_on, "case-a.khan")
print("   M05       ->", r1["report"])
print("   follow-up ->", r2["report"])
print("   escalations on this thread:", len(mem_on.get_state(case).values["escalations"]),
      "| checkpoints saved:", len(list(mem_on.get_state_history(case))))

print("\n3) the same follow-up alone, on a NEW thread, memory branch ON")
fresh = {"configurable": {"thread_id": "case-new"}}
print("   case-new has state before the run?", bool(mem_on.get_state(fresh).values))
r3 = mem_on.invoke(blank(follow_up), fresh)
print("   follow-up ->", r3["report"])

print("\nhow the follow-up moved through the graph on the open case:")
case2 = {"configurable": {"thread_id": "case-a.khan-trace"}}
mem_on.invoke(blank(by_id["M05"]["alert"]), case2)
for step in mem_on.stream({"alert": follow_up, "risk": "", "cve_info": "", "report": ""}, case2,
                          stream_mode="updates"):
    for node, update in step.items():
        print("  -> %-12s returned %s" % (node, {k: (v if k != "history" else "[...]") for k, v in update.items()}))
KEYWORDS[:] = LECTURE

1) same thread, memory branch OFF
   M05       -> ESCALATE. No specific CVE found; follow standard triage.
   follow-up -> CLOSE. Logged for monitoring; no action needed.
   the thread remembers 2 alerts:
     1. Entra ID sign-in for a.khan from Lagos 25 minutes after a sign-in from Cincinnati, flagged as i
     2. Exchange audit: new inbox rule on a.khan's mailbox forwards all mail to an external address, cr

2) same thread, memory branch ON
   M05       -> ESCALATE. No specific CVE found; follow standard triage.
   follow-up -> ESCALATE (open case). No keyword, but this thread already holds 1 escalated alert(s): Entra ID sign-in for a.khan from Lagos 25 minutes 
   escalations on this thread: 2 | checkpoints saved: 10

3) the same follow-up alone, on a NEW thread, memory branch ON
   case-new has state before the run? False
   follow-up -> CLOSE. Logged for monitoring; no action needed.

how the follow-up moved through the graph on the open case:
  -> classify     returned {'risk': '

This is the clearest evidence for the report. The same follow-up alert is **closed** on a fresh
thread and **escalated** on the open case, and the only difference is what the checkpointer saved
under the `thread_id`. The decision depends on earlier runs, which a chain, which starts from nothing
on every call, cannot do. The cost is that the decision now depends on how alerts are grouped into
threads: put the wrong alerts on one thread and the memory branch over-escalates.

### Summary table (`results_mine.csv`)

In [18]:
print("%-4s %-19s %-34s %-7s %-7s %-6s %s" % ("row", "keyword list", "graph", "caught", "missed", "over", "one change from the row it builds on"))
for r in results:
    print("%-4d %-19s %-34s %d/6    %d/6    %d/6    %s" % (r["step"], r["keyword_list"], r["graph"],
          r["caught"], r["missed"], r["over_escalated"], r["one_change"]))

row  keyword list        graph                              caught  missed  over   one change from the row it builds on
1    narrow (2 words)    lecture graph                      1/6    5/6    0/6    remove exfil, critical, admin from the lecture list
2    lecture (5 words)   lecture graph                      2/6    4/6    3/6    baseline: the Lecture 3 rule
3    wider (8 words)     lecture graph                      4/6    2/6    3/6    add cleared, powershell, encoded (for missed M02, M04)
4    widest (13 words)   lecture graph                      6/6    0/6    5/6    add travel, dns, password, scan, test (for missed M05, M06)
5    widest (13 words)   + lookup_asset (first host named)  6/6    0/6    5/6    graph: add context node, lookup_asset tool, new router
6    widest (13 words)   + lookup_asset (host it is on)     6/6    0/6    4/6    host = the asset the alert happened on, not the first named
7    widest (13 words)   + check_change (both tools)        6/6    0/6    1/6    gr

---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_Agent.md` for the full
requirements. The questions that matter:

- The two errors are not equal. Which would you rather your agent made, and why? Answer as the
  person who has to staff the team.
- Which rule would you deploy, and what does it cost you? Name the alerts you are choosing to get
  wrong.
- Some alerts stay wrong at every setting. Explain why in terms of what a keyword rule can and
  cannot see, and describe the change you made to the graph and what it bought you.
- What did the graph give you that a LangChain chain could not? Point at your conditional edge and
  at `thread_id` memory, using your own output as evidence, and say when a chain would be the better
  choice.
- Twelve alerts means one alert is a sixth of a column. How much confidence does that justify, and
  what would you measure next?

Submit one PDF report with the link to your project folder and the link to your video on its first
page. The notebook, `my_alerts.json`, and your results live in the repository, which is private, with
your instructor and the TA added as collaborators. The notebook must be committed with its outputs
showing.